# RedSpell: data, training, evaluation
Pipeline: **1** data → **2** preprocessing → **3** representation → **4** modeling → **5** evaluation.

This notebook so far: setup (section 0) and the dataset (section 1).

## 0. Setup

In [ ]:
!pip -q install datasets transformers sentencepiece accelerate nltk sacremoses
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE -> Runtime > Change runtime type > T4 GPU")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/redspell')   # everything persistent lives here
DATA, CKPT, MODELS = ROOT/'data', ROOT/'checkpoints', ROOT/'models'
for p in (DATA, CKPT, MODELS):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile generate_errors.py
"""Synthetic error injection for English grammar/spelling correction (RedSpell).

Takes clean sentences and returns (incorrect, correct) pairs.
Usage:
    from src.data.generate_errors import make_pair, generate_pairs
"""
import random
import re
from typing import Iterable, Iterator, Optional

# --- error resources ---------------------------------------------------------
KEYBOARD = {
    "a": "qwsz", "b": "vghn", "c": "xdfv", "d": "serfcx", "e": "wsdr", "f": "drtgvc",
    "g": "ftyhbv", "h": "gyujnb", "i": "ujko", "j": "huikmn", "k": "jiolm", "l": "kop",
    "m": "njk", "n": "bhjm", "o": "iklp", "p": "ol", "q": "wa", "r": "edft", "s": "awedxz",
    "t": "rfgy", "u": "yhji", "v": "cfgb", "w": "qase", "x": "zsdc", "y": "tghu", "z": "asx",
}

_CONFUSION_PAIRS = [
    ("their", "there"), ("your", "you're"), ("its", "it's"), ("then", "than"),
    ("to", "too"), ("affect", "effect"), ("lose", "loose"), ("were", "where"),
    ("whether", "weather"), ("accept", "except"), ("principal", "principle"),
]
CONFUSIONS = {}
for a, b in _CONFUSION_PAIRS:
    CONFUSIONS.setdefault(a, []).append(b)
    CONFUSIONS.setdefault(b, []).append(a)

AGREEMENT = {"is": "are", "are": "is", "was": "were", "were": "was",
             "has": "have", "have": "has", "does": "do", "do": "does"}
ARTICLES = {"a", "an", "the"}
PREPOSITIONS = {"in": ["on", "at"], "on": ["in", "at"], "at": ["in", "on"],
                "of": ["for", "from"], "for": ["of", "to"], "with": ["by", "of"]}

_TOKEN_RE = re.compile(r"\s+|[A-Za-z']+|[^A-Za-z'\s]+")


def _is_word(tok: str) -> bool:
    return bool(tok) and (tok[0].isalpha() or tok[0] == "'") and not tok.isspace()


def _match_case(src: str, new: str) -> str:
    if src.isupper() and len(src) > 1:
        return new.upper()
    if src[0].isupper():
        return new[0].upper() + new[1:]
    return new


def _word_idx(tokens: list) -> list:
    return [i for i, t in enumerate(tokens) if _is_word(t)]


# --- individual error operations (mutate `tokens`, return True if applied) ----
def op_typo(tokens, rng):
    cands = [i for i in _word_idx(tokens) if len(tokens[i]) >= 4 and tokens[i].isalpha()]
    if not cands:
        return False
    i = rng.choice(cands)
    w = tokens[i]
    p = rng.randrange(1, len(w) - 1)  # never touch first/last char: keeps typos realistic
    kind = rng.choice(["swap", "delete", "duplicate", "keyboard"])
    if kind == "swap":
        new = w[:p] + w[p + 1] + w[p] + w[p + 2:]
    elif kind == "delete":
        new = w[:p] + w[p + 1:]
    elif kind == "duplicate":
        new = w[:p] + w[p] + w[p:]
    else:
        nb = KEYBOARD.get(w[p].lower())
        if not nb:
            return False
        r = rng.choice(nb)
        new = w[:p] + (r.upper() if w[p].isupper() else r) + w[p + 1:]
    if new == w:
        return False
    tokens[i] = new
    return True


def op_confusion(tokens, rng):
    cands = [i for i in _word_idx(tokens) if tokens[i].lower() in CONFUSIONS]
    if not cands:
        return False
    i = rng.choice(cands)
    tokens[i] = _match_case(tokens[i], rng.choice(CONFUSIONS[tokens[i].lower()]))
    return True


def op_agreement(tokens, rng):
    cands = [i for i in _word_idx(tokens) if tokens[i].lower() in AGREEMENT]
    if not cands:
        return False
    i = rng.choice(cands)
    tokens[i] = _match_case(tokens[i], AGREEMENT[tokens[i].lower()])
    return True


def op_article(tokens, rng):
    cands = [i for i in _word_idx(tokens) if tokens[i].lower() in ARTICLES]
    if not cands:
        return False
    i = rng.choice(cands)
    w = tokens[i].lower()
    if w in {"a", "an"} and rng.random() < 0.4:
        tokens[i] = _match_case(tokens[i], "an" if w == "a" else "a")
        return True
    # drop the article (and the space after it)
    if i + 1 < len(tokens) and tokens[i + 1].isspace():
        del tokens[i:i + 2]
        if i == 0 and tokens and tokens[0][0].isalpha():  # keep sentence capitalisation
            tokens[0] = tokens[0][0].upper() + tokens[0][1:]
        return True
    return False


def op_preposition(tokens, rng):
    cands = [i for i in _word_idx(tokens) if tokens[i].lower() in PREPOSITIONS]
    if not cands:
        return False
    i = rng.choice(cands)
    tokens[i] = _match_case(tokens[i], rng.choice(PREPOSITIONS[tokens[i].lower()]))
    return True


def op_repeat(tokens, rng):
    cands = [i for i in _word_idx(tokens) if tokens[i].isalpha() and len(tokens[i]) <= 6]
    if not cands:
        return False
    i = rng.choice(cands)
    tokens[i:i + 1] = [tokens[i], " ", tokens[i].lower()]
    return True


# (name, function, sampling weight)
OPS = [
    ("typo", op_typo, 0.35),
    ("confusion", op_confusion, 0.15),
    ("agreement", op_agreement, 0.15),
    ("article", op_article, 0.15),
    ("preposition", op_preposition, 0.12),
    ("repeat", op_repeat, 0.08),
]


def make_pair(sentence: str, rng: random.Random,
              n_errors: Optional[int] = None) -> Optional[dict]:
    """Return {'source': incorrect, 'target': sentence, 'errors': [...]} or None."""
    tokens = _TOKEN_RE.findall(sentence)
    if "".join(tokens) != sentence:  # tokenizer round-trip must be lossless
        return None
    if n_errors is None:
        n_errors = rng.choices([1, 2, 3], weights=[0.5, 0.35, 0.15])[0]
    names = [o[0] for o in OPS]
    funcs = {o[0]: o[1] for o in OPS}
    weights = [o[2] for o in OPS]
    applied = []
    for _ in range(n_errors * 3):  # a few retries: some ops don't apply to a sentence
        if len(applied) >= n_errors:
            break
        name = rng.choices(names, weights=weights)[0]
        if funcs[name](tokens, rng):
            applied.append(name)
    source = "".join(tokens)
    if not applied or source == sentence:
        return None
    return {"source": source, "target": sentence, "errors": applied}


def generate_pairs(sentences: Iterable[str], rng: random.Random,
                   identity_ratio: float = 0.05) -> Iterator[dict]:
    """Yield pairs. `identity_ratio` of them are unchanged (source == target) so the
    model learns to leave correct text alone."""
    for s in sentences:
        if rng.random() < identity_ratio:
            yield {"source": s, "target": s, "errors": []}
            continue
        pair = make_pair(s, rng)
        if pair:
            yield pair


if __name__ == "__main__":
    demo = [
        "The students are going to their classroom after the lecture.",
        "She has finished the report and sent it to the manager.",
        "We need to decide whether the weather is good enough for a trip.",
    ]
    r = random.Random(0)
    for s in demo:
        print(make_pair(s, r))


In [ ]:
from generate_errors import make_pair, generate_pairs

In [ ]:
SEED = 42
N_PAIRS = 80_000        # brief asks for >= 50k; margin for filtering
MIN_WORDS, MAX_WORDS = 8, 35
IDENTITY_RATIO = 0.05   # share of unchanged pairs (teaches the model not to over-correct)

# --- real data (cLang-8) ---
# Put the TSV produced by cLang-8's run.sh in Drive: redspell/data/ (adjust the filename)
CLANG8_TSV = DATA / 'clang8_source_target_en.tsv'
N_REAL = 100_000        # real training pairs to keep
N_REAL_HELD = 3_000     # real val pairs AND real test pairs (never trained on)

## 1. Dataset
Clean English sentences from C4 (streamed, no big download), then synthetic error injection. Lang-8 pairs can be added later as real-error data.

In [ ]:
import re, random, nltk
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab', quiet=True)

def keep(s):
    n = len(s.split())
    if not (MIN_WORDS <= n <= MAX_WORDS) or not s.isascii():
        return False
    if not s[0].isupper() or s[-1] not in '.!?':
        return False
    if re.search(r'http|www\.|@|\d{4,}|[{}<>\[\]|\\_*#]', s):
        return False
    return sum(c.isalpha() or c == ' ' for c in s) / len(s) > 0.9

stream = load_dataset('allenai/c4', 'en', split='train', streaming=True)
clean, seen = [], set()
for doc in stream:
    for s in sent_tokenize(doc['text']):
        s = s.strip()
        if s not in seen and keep(s):
            seen.add(s); clean.append(s)
    if len(clean) >= N_PAIRS:
        break
clean = clean[:N_PAIRS]
print(len(clean), 'clean sentences'); print(*clean[:3], sep='\n')

In [ ]:
import pandas as pd

rng = random.Random(SEED)
df = pd.DataFrame(generate_pairs(clean, rng, IDENTITY_RATIO))
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n = len(df); i1, i2 = int(.90 * n), int(.95 * n)
splits = {'train': df[:i1], 'val': df[i1:i2], 'test': df[i2:]}
for name, part in splits.items():
    part.to_json(DATA / f'synth_{name}.jsonl', orient='records', lines=True)
    print(name, len(part))

# for the report: error-type distribution and a few examples
print(df['errors'].explode().value_counts())
df.sample(8, random_state=1)[['source', 'target']]

## 1b. Real data: cLang-8 (optional until the file is available)
Real learner sentences. Skipped automatically if the TSV is not in Drive yet, so the notebook still runs on synthetic data only.

In [ ]:
import re
import pandas as pd

if not CLANG8_TSV.exists():
    print('cLang-8 TSV not found -> synthetic-only run. Expected at:', CLANG8_TSV)
else:
    real = pd.read_csv(CLANG8_TSV, sep='\t', header=None, names=['source', 'target'],
                       quoting=3, on_bad_lines='skip', dtype=str).dropna()
    real = real[real.source.str.strip().ne('') & real.target.str.strip().ne('')]

    # cLang-8's default output is spaCy-tokenized ('is n't gramatical .'); the app receives
    # natural text, so detokenize to match it.
    probe = real.source.sample(min(len(real), 2000), random_state=0)
    if probe.str.contains(r" n't| [.,!?]$").mean() > 0.3:
        from sacremoses import MosesDetokenizer
        md = MosesDetokenizer(lang='en')
        detok = lambda s: re.sub(r" n't\b", "n't", md.detokenize(s.split()))
        real['source'] = real.source.map(detok)
        real['target'] = real.target.map(detok)
        print('detokenized')

    wc = lambda s: s.str.split().str.len()
    real = real[wc(real.source).between(3, 50) & wc(real.target).between(3, 50)]
    real = real[(wc(real.target) / wc(real.source)).between(0.5, 2.0)]   # drop misaligned pairs
    real = real.drop_duplicates('source').sample(frac=1, random_state=SEED)
    real = real.head(N_REAL + 2 * N_REAL_HELD).reset_index(drop=True)

    parts = {'test': real[:N_REAL_HELD],
             'val': real[N_REAL_HELD:2 * N_REAL_HELD],
             'train': real[2 * N_REAL_HELD:]}
    for name, part in parts.items():
        part.to_json(DATA / f'real_{name}.jsonl', orient='records', lines=True)
        print(name, len(part))
    print('unchanged pairs: %.1f%%' % (100 * (real.source == real.target).mean()))
    display(real.sample(8, random_state=1))

## 1c. Final splits
Train/val mix real and synthetic. Two separate test sets: **real** (the number that matters) and **synthetic** (for comparison in the report).

In [ ]:
cols = ['source', 'target', 'origin']

def load(name, origin):
    p = DATA / f'{name}.jsonl'
    if not p.exists():
        return pd.DataFrame(columns=cols)
    d = pd.read_json(p, lines=True, dtype=False)
    d['origin'] = origin
    return d[cols]

train = pd.concat([load('real_train', 'real'), load('synth_train', 'synth')])
train = train.sample(frac=1, random_state=SEED).reset_index(drop=True)
val = pd.concat([load('real_val', 'real'), load('synth_val', 'synth')]).reset_index(drop=True)
test_real, test_synth = load('real_test', 'real'), load('synth_test', 'synth')

for name, d in {'train': train, 'val': val, 'test_real': test_real, 'test_synth': test_synth}.items():
    d.to_json(DATA / f'{name}.jsonl', orient='records', lines=True)
    print(name, len(d), dict(d.origin.value_counts()))